# 35 / 37 / 38 / 39 / 41 — Expériences de graines (seed lottery)

Le pipeline champion+pseudo+rank-w50 décliné par graine, et les croisements champion-seedX × pseudo-seedY.
La CV 3-folds montre que l'écart inter-graines (~±0.0002) est du **bruit** — mais sur le public 30%,
certains tirages payent : **39_c123x23_w50 → LB 0.358104** (meilleur public de l'équipe),
38_blend_c42xs7_w35 → 0.357920, contre 35_seed5 → 0.352731 (loterie assumée, le LB garde le meilleur).
⚠️ Long à exécuter (~14 entraînements CatBoost).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from catboost import CatBoostClassifier
from src import config as C
from src.utils import op03_mask, seed_everything, make_submission
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv"); test = pd.read_csv(DATA / "test.csv")
op03 = op03_mask(train).to_numpy(); y_all = train[C.TARGET].to_numpy()
te_op = op03_mask(test).to_numpy()
EPS = 1e-6; W = (5, 10, 20); SM = 30
def rowf(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]; f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)
def bb(df, ref):
    X = rowf(df).reset_index(drop=True)
    for c in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        X[f"fq_{c}"] = df[c].map(ref[c].value_counts(normalize=True)).fillna(0).values
    return pd.concat([X, behavioral_features(df, ref).reset_index(drop=True),
                      recency_features(df, ref).reset_index(drop=True),
                      recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, W).reset_index(drop=True)], axis=1)
def ftr(df, ref):
    X = bb(df, ref); X["te"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM); return X
def fap(df, ref):
    X = bb(df, ref); mp, gm = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM)
    X["te"] = apply_target_map(df, C.ORIGIN_ACCT, mp, gm); return X
def cat(seed=42):
    return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                              learning_rate=0.05, iterations=600, random_seed=seed, verbose=False)
def rk(x):
    return np.argsort(np.argsort(x)) / (len(x) - 1)
def out(name, p_op):
    full = np.zeros(len(test)); full[te_op] = p_op
    print("écrit :", make_submission(test[C.ID], full, name))
ref0 = train.iloc[np.where(op03)[0]]; y0 = y_all[op03]
test_op = test.iloc[np.where(te_op)[0]].copy()
print("setup OK |", op03.sum(), "op03 train /", te_op.sum(), "op03 test")

In [ ]:
def train_champion(seed=42):
    """Champion CatBoost (config nb08) entraîné sur tout le train op_03."""
    return cat(seed).fit(ftr(ref0, ref0), y0).predict_proba(fap(test_op, ref0))[:, 1]

def pseudo_from(pch, thr_f=0.98, thr_l=0.02, seed=42):
    """Pseudo-labeling 1 cycle (cf. nb23) puis réentraînement."""
    mfm = pch > thr_f; mlm = pch < thr_l
    pse = test_op.iloc[np.where(mfm | mlm)[0]].copy()
    pse[C.TARGET] = (pch[mfm | mlm] > thr_f).astype(float)
    aug = pd.concat([ref0, pse], ignore_index=True)
    print(f"pseudo ({thr_f}/{thr_l}, seed {seed}) : {int(mfm.sum())} fraudes / {int(mlm.sum())} légitimes")
    m = cat(seed).fit(ftr(aug, aug), aug[C.TARGET].to_numpy())
    return m.predict_proba(fap(test_op, aug))[:, 1]

## Champions et pseudos par graine

In [ ]:
SEEDS = [42, 5, 7, 11, 99, 123, 777]
champ, pseu = {}, {}
for s in SEEDS:
    champ[s] = train_champion(seed=s)
    pseu[s] = pseudo_from(champ[s], seed=s)
    print("seed", s, "ok")

## 35 / 37 — pipeline complet par graine (rank w50)

In [ ]:
out("35_seed5_rank_w50", 0.5 * rk(champ[5]) + 0.5 * rk(pseu[5]))
for s in [7, 11, 99, 123, 777]:
    out(f"37_seed{s}_rank_w50", 0.5 * rk(champ[s]) + 0.5 * rk(pseu[s]))

## 38 — champion seed42 × pseudo seed7

In [ ]:
out("38_blend_c42xs7_w35", 0.65 * rk(champ[42]) + 0.35 * rk(pseu[7]))
out("38_blend_c42xs7_w50", 0.50 * rk(champ[42]) + 0.50 * rk(pseu[7]))

## 39 — croisements champion-seedX × pseudo-seedY (rank w50)

In [ ]:
for s in [7, 11, 99, 123, 777]:
    out(f"39_08xp{s}_w50", 0.5 * rk(champ[42]) + 0.5 * rk(pseu[s]))
    out(f"39_c{s}x23_w50", 0.5 * rk(champ[s]) + 0.5 * rk(pseu[42]))

## 41 — balayage de poids autour du meilleur croisement (c123 × pseudo42)

In [ ]:
for wp in [42, 55, 60]:
    out(f"41_c123x23_w{wp}", (1 - wp / 100) * rk(champ[123]) + (wp / 100) * rk(pseu[42]))